# Premier League 2025/26 — Data Analysis

This notebook analyses match results, goals, shots, and xG stats fetched from [Understat](https://understat.com).

**Run `python fetch_pl_data.py` first** (optionally with `--shots`) to populate the `data/` folder with live CSVs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

DATA_DIR = Path('data')

# Load data — falls back to sample files if live data not yet fetched
def load(filename, fallback):
    path = DATA_DIR / filename
    if path.exists():
        return pd.read_csv(path)
    print(f'Using sample data ({fallback}). Run fetch_pl_data.py to get live data.')
    return pd.read_csv(DATA_DIR / fallback)

matches = load('pl_2025_26_results.csv', 'sample_results.csv')
teams   = load('pl_2025_26_team_stats.csv', 'sample_team_stats.csv')

print(f'Matches loaded : {len(matches)}')
print(f'Teams loaded   : {len(teams)}')
matches.head()

## League Table

In [ ]:
cols = ['team', 'played', 'wins', 'draws', 'losses',
        'goals_scored', 'goals_conceded', 'goal_diff',
        'xg_for', 'xg_against', 'xg_diff', 'points']
teams[cols].style \
    .background_gradient(subset=['points'], cmap='RdYlGn') \
    .background_gradient(subset=['xg_diff'], cmap='RdYlGn') \
    .format({'xg_for': '{:.2f}', 'xg_against': '{:.2f}', 'xg_diff': '{:.2f}'})

## Goals vs xG per Team

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(teams))
ax.bar([i - 0.2 for i in x], teams['goals_scored'], width=0.4, label='Goals Scored', color='steelblue')
ax.bar([i + 0.2 for i in x], teams['xg_for'], width=0.4, label='xG For', color='orange', alpha=0.8)

ax.set_xticks(list(x))
ax.set_xticklabels(teams['team'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Goals / xG')
ax.set_title('Premier League 2025/26 — Goals Scored vs xG For')
ax.legend()
plt.tight_layout()
plt.show()

## xG Diff (xG For − xG Against)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['green' if v >= 0 else 'red' for v in teams['xg_diff']]
ax.barh(teams['team'][::-1], teams['xg_diff'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('xG Difference (xGF − xGA)')
ax.set_title('Premier League 2025/26 — xG Difference by Team')
plt.tight_layout()
plt.show()

## Match Results — Goals & xG Timeline

In [ ]:
matches['total_goals'] = matches['home_goals'] + matches['away_goals']
matches['total_xg']   = matches['home_xg'] + matches['away_xg']

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(matches.index, matches['total_goals'], marker='o', label='Goals', color='steelblue', linewidth=1.2, markersize=4)
ax.plot(matches.index, matches['total_xg'],   marker='s', label='xG',    color='orange',    linewidth=1.2, markersize=4, linestyle='--')
ax.set_xlabel('Match #')
ax.set_ylabel('Total per match')
ax.set_title('Premier League 2025/26 — Goals & xG per Match')
ax.legend()
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

print(f'Avg goals/match : {matches["total_goals"].mean():.2f}')
print(f'Avg xG/match    : {matches["total_xg"].mean():.2f}')

## Shot Analysis (requires `--shots` flag)

In [ ]:
shots_path = DATA_DIR / 'pl_2025_26_shots.csv'
sample_shots_path = DATA_DIR / 'sample_shots.csv'
shots = pd.read_csv(shots_path if shots_path.exists() else sample_shots_path)
print(f'Shots loaded: {len(shots)}')
shots.head()

In [ ]:
# Shot outcome distribution
outcome_counts = shots['result'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
outcome_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Shot Outcomes — Premier League 2025/26')
ax.set_xlabel('Outcome')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# xG distribution per shot
shots['xg'] = pd.to_numeric(shots['xg'], errors='coerce')
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(shots['xg'].dropna(), bins=20, color='steelblue', edgecolor='white')
ax.set_title('xG Distribution per Shot')
ax.set_xlabel('xG')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f'Mean xG per shot : {shots["xg"].mean():.3f}')
print(f'Shots on target  : {shots[shots["result"].isin(["Goal","SavedShot"])].shape[0]}')

In [ ]:
# Situation breakdown
sit = shots.groupby('situation')['xg'].agg(['count', 'sum', 'mean']).round(3)
sit.columns = ['shots', 'total_xg', 'avg_xg']
sit.sort_values('shots', ascending=False)